## Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  str    
 3   color       53943 non-null  str    
 4   clarity     53943 non-null  str    
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), str(3)
memory usage: 4.5 MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2

## Tiền xử lý dữ liệu

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


## Chia tập dữ liệu

In [3]:
from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(n_splits = 5, shuffle = True, random_state=42)

## Traning và kết quả đánh giá

In [4]:

# Import mô hình cây quyết định hồi quy
from sklearn.tree import DecisionTreeRegressor
# Import các hàm đánh giá mô hình hồi quy
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import thư viện tính toán số học
import numpy as np
# Import thư viện xử lý DataFrame
import pandas as pd
# Import thư viện đo thời gian
import time

# Khai báo các giá trị max_depth cần thử (từ 1 đến 12)
max_depth_values = range(1, 13)
# Khai báo các giá trị min_samples_leaf cần thử (từ 1 đến 8)
min_samples_leaf_values = range(1, 9)
# Khai báo các giá trị min_samples_split cần thử (từ 2 đến 12)
min_samples_split_values = range(2, 13)

# Khởi tạo RMSE Test trung bình tốt nhất bằng vô cùng
best_rmse = float('inf')
# Khởi tạo biến lưu bộ tham số tốt nhất
best_params = None
# Khởi tạo biến lưu kết quả từng fold của bộ tham số tốt nhất
best_fold_results = None
# Khởi tạo biến lưu tổng thời gian huấn luyện 5 fold của bộ tham số tốt nhất
best_times = None

# Ghi nhận thời điểm bắt đầu tìm kiếm tham số
search_start_time = time.perf_counter()

# Duyệt qua từng giá trị max_depth
for max_depth in max_depth_values:
  # Duyệt qua từng giá trị min_samples_leaf
  for min_samples_leaf in min_samples_leaf_values:
    # Duyệt qua từng giá trị min_samples_split
    for min_samples_split in min_samples_split_values:


      # Khởi tạo danh sách lưu kết quả Train
      mae_train_all = []
      mse_train_all = []
      rmse_train_all = []
      r2_train_all = []

      # Khởi tạo danh sách lưu kết quả Test
      mae_test_all = []
      mse_test_all = []
      rmse_test_all = []
      r2_test_all = []

      # Khởi tạo danh sách lưu số mẫu và thời gian
      train_samples_all = []
      test_samples_all = []
      times_all = []

      # Vòng lặp qua 5 fold
      for fold, (train_index, test_index) in enumerate(Kf.split(X), start=1):

        # Lấy dữ liệu Train
        X_train = X.iloc[train_index]
        y_train = y.iloc[train_index]

        # Lấy dữ liệu Test
        X_test = X.iloc[test_index]
        y_test = y.iloc[test_index]

        # Khởi tạo mô hình với bộ tham số hiện tại
        model = DecisionTreeRegressor(
            criterion="squared_error",
            random_state=42,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split
        )

        # Bắt đầu đo thời gian huấn luyện
        start_time = time.perf_counter()

        # Huấn luyện mô hình trên tập Train của fold hiện tại
        model.fit(X_train, y_train)

        # Tính thời gian huấn luyện của fold hiện tại
        training_time = time.perf_counter() - start_time

        # Lưu thời gian huấn luyện
        times_all.append(training_time)

        # Dự đoán trên tập Train
        y_pred_train = model.predict(X_train)

        # Dự đoán trên tập Test
        y_pred_test = model.predict(X_test)

        # Tính MAE trên Train
        mae_train = mean_absolute_error(y_train, y_pred_train)

        # Tính MSE trên Train
        mse_train = mean_squared_error(y_train, y_pred_train)

        # Tính RMSE trên Train
        rmse_train = np.sqrt(mse_train)

        # Tính R² trên Train
        r2_train = r2_score(y_train, y_pred_train)

        # Tính MAE trên Test
        mae_test = mean_absolute_error(y_test, y_pred_test)

        # Tính MSE trên Test
        mse_test = mean_squared_error(y_test, y_pred_test)

        # Tính RMSE trên Test
        rmse_test = np.sqrt(mse_test)

        # Tính R² trên Test
        r2_test = r2_score(y_test, y_pred_test)

        # Lưu kết quả Train
        mae_train_all.append(mae_train)
        mse_train_all.append(mse_train)
        rmse_train_all.append(rmse_train)
        r2_train_all.append(r2_train)

        # Lưu kết quả Test
        mae_test_all.append(mae_test)
        mse_test_all.append(mse_test)
        rmse_test_all.append(rmse_test)
        r2_test_all.append(r2_test)

        # Lưu số mẫu Train và Test
        train_samples_all.append(len(train_index))
        test_samples_all.append(len(test_index))

      # Tính RMSE Test trung bình của 5 fold để chọn tham số
      mean_rmse_test = np.mean(rmse_test_all)

      # Kiểm tra RMSE Test trung bình có tốt hơn không
      if mean_rmse_test < best_rmse:

        # Cập nhật RMSE Test trung bình tốt nhất
        best_rmse = mean_rmse_test

        # Lưu bộ tham số tốt nhất
        best_params = {
          'max_depth': max_depth,
          'min_samples_leaf': min_samples_leaf,
          'min_samples_split': min_samples_split
        }

        # Lưu tổng thời gian huấn luyện 5 fold
        best_times = sum(times_all)

        # Khởi tạo danh sách lưu kết quả từng fold tốt nhất
        best_fold_results = []

        # Lưu kết quả đánh giá của từng fold
        for i in range(5):
          best_fold_results.append({
            'Fold': f'Fold {i + 1}',
            'Số mẫu Train': train_samples_all[i],
            'Số mẫu Test': test_samples_all[i],
            'MAE Train': mae_train_all[i],
            'MSE Train': mse_train_all[i],
            'RMSE Train': rmse_train_all[i],
            'R² Train': r2_train_all[i],
            'MAE Test': mae_test_all[i],
            'MSE Test': mse_test_all[i],
            'RMSE Test': rmse_test_all[i],
            'R² Test': r2_test_all[i],
            'Thời gian (giây)': times_all[i]
          })

# Tính tổng thời gian tìm kiếm 200 tổ hợp tham số
search_time = time.perf_counter() - search_start_time

# Chuyển kết quả từng fold thành DataFrame
results_df = pd.DataFrame(best_fold_results)

# Tạo dòng kết quả trung bình của 5 fold
mean_results = {
  'Fold': 'Trung bình',
  'Số mẫu Train': results_df['Số mẫu Train'].mean(),
  'Số mẫu Test': results_df['Số mẫu Test'].mean(),
  'MAE Train': results_df['MAE Train'].mean(),
  'MSE Train': results_df['MSE Train'].mean(),
  'RMSE Train': results_df['RMSE Train'].mean(),
  'R² Train': results_df['R² Train'].mean(),
  'MAE Test': results_df['MAE Test'].mean(),
  'MSE Test': results_df['MSE Test'].mean(),
  'RMSE Test': results_df['RMSE Test'].mean(),
  'R² Test': results_df['R² Test'].mean(),
  'Thời gian (giây)': results_df['Thời gian (giây)'].mean()
}

# Thêm dòng trung bình vào bảng kết quả
results_df = pd.concat(
    [results_df, pd.DataFrame([mean_results])],
    ignore_index=True
)

# In bộ tham số tốt nhất
print("Bộ tham số tốt nhất:", best_params)

# In RMSE Test trung bình tốt nhất
print(f"RMSE Test trung bình tốt nhất: {best_rmse:.4f}")

# In kết quả Train và Test của từng fold và trung bình
print("\nKết quả đánh giá của bộ tham số tốt nhất:")
print(results_df.round(4).to_string(index=False))

# In tổng thời gian huấn luyện 5 fold
print(f"\nTổng thời gian huấn luyện 5 fold: {best_times:.6f} giây")

# In tổng thời gian tìm kiếm 200 tổ hợp tham số
print(f"Tổng thời gian tìm kiếm: {search_time:.4f} giây")

Bộ tham số tốt nhất: {'max_depth': 12, 'min_samples_leaf': 8, 'min_samples_split': 2}
RMSE Test trung bình tốt nhất: 603.4491

Kết quả đánh giá của bộ tham số tốt nhất:
      Fold  Số mẫu Train  Số mẫu Test  MAE Train   MSE Train  RMSE Train  R² Train  MAE Test    MSE Test  RMSE Test  R² Test  Thời gian (giây)
    Fold 1       43020.0      10755.0   261.0278 247811.2057    497.8064    0.9844  310.7850 346321.6548   588.4910   0.9781            0.1091
    Fold 2       43020.0      10755.0   260.5761 244351.3231    494.3190    0.9847  307.8532 347074.9428   589.1307   0.9777            0.1119
    Fold 3       43020.0      10755.0   259.2647 242546.5632    492.4902    0.9847  305.1828 354483.2700   595.3850   0.9776            0.1094
    Fold 4       43020.0      10755.0   261.6677 246930.3994    496.9209    0.9844  315.5531 367917.3970   606.5619   0.9774            0.1167
    Fold 5       43020.0      10755.0   255.7908 232856.1400    482.5517    0.9853  315.6882 406631.6597   637.6768 